# 00RecoverSourceState

Recover source state and rebuild the reference-only observed package

This notebook is orchestration-only. It cannot enable empirical analysis or production promotion.

In [ ]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A1-SOURCE"
STAGE_ID = "A1-00-RECOVER"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "source-policy-run-v1"
DATA_VERSION = "OBSERVED_DEV_20260806_01"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "crawl/releases/CRAWL_20260806_03/HANDOFF.json"
OUTPUT_ROOT = "crawl/runs/notebooks/observed-dev/AGENT1_20260806_01"
RANDOM_SEED = 42
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

## Imports and isolated observed-development environment

In [ ]:
from pathlib import Path
import sys

def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "crawl" / "src" / "p4_crawl").is_dir():
            return candidate
        nested = candidate / "DSJA" / "project_4"
        if (nested / "crawl" / "src" / "p4_crawl").is_dir():
            return nested
    raise RuntimeError("Could not locate DSJA/project_4")

PROJECT_ROOT = locate_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "crawl" / "src"))
from p4_crawl.config import RunConfig
from p4_crawl.observed import require_repository_relative

assert RUN_MODE == "observed-dev"
assert CONTRACT_VERSION == "2.1.2"
assert CRAWL_RELEASE_ID == "CRAWL_20260806_03"
assert EMPIRICAL_ANALYSIS_ALLOWED is False
assert Path(OUTPUT_ROOT).as_posix().startswith("crawl/runs/notebooks/observed-dev/")
require_repository_relative(INPUT_MANIFEST_PATH)
require_repository_relative(OUTPUT_ROOT)

CRAWL_ROOT = PROJECT_ROOT / "crawl"
RUN_ROOT = PROJECT_ROOT / OUTPUT_ROOT
STAGE_ROOT = RUN_ROOT / STAGE_ID
INPUT_MANIFEST = PROJECT_ROOT / INPUT_MANIFEST_PATH
run_id = Path(OUTPUT_ROOT).relative_to("crawl/runs").as_posix()
config = RunConfig(
    project_root=PROJECT_ROOT, run_id=run_id, run_mode=RUN_MODE,
    contract_version=CONTRACT_VERSION, crawl_release_id=CRAWL_RELEASE_ID,
    data_version=DATA_VERSION, as_of_date=AS_OF_DATE, random_seed=RANDOM_SEED,
)
PARAMETERS = {name: globals()[name] for name in [
    "RUN_MODE", "AGENT_ID", "STAGE_ID", "CONTRACT_VERSION", "SCHEMA_VERSION",
    "DATA_VERSION", "CRAWL_RELEASE_ID", "AS_OF_DATE", "INPUT_MANIFEST_PATH",
    "OUTPUT_ROOT", "RANDOM_SEED", "FAIL_ON_GATE", "EMPIRICAL_ANALYSIS_ALLOWED",
]}

def quality_row(gate, rule, severity, status, observed, threshold, evidence):
    return {
        "gateId": gate, "ruleId": rule, "severity": severity, "status": status,
        "observedValue": observed, "threshold": threshold,
        "evidencePath": f"{OUTPUT_ROOT}/{STAGE_ID}/{evidence}",
    }

## Input and checksum audit

In [ ]:
from p4_crawl.observed import audit_input_manifest

input_audit = audit_input_manifest(PROJECT_ROOT, INPUT_MANIFEST_PATH, CRAWL_RELEASE_ID)
assert input_audit["contractVersion"] == CONTRACT_VERSION
input_audit

## Stage module call

In [ ]:
STAGE_WARNING = 'Observed input is partial and cannot be promoted to empirical analysis.'
from p4_crawl.release import build_observed_input_package, recover_source_state

recovery = recover_source_state(config, STAGE_ROOT)
package_root = STAGE_ROOT / "observed_package"
package = build_observed_input_package(config, package_root)
metrics = {**recovery, "observedPackagePostingRows": package["postingRows"], "observedPackageRawHtmlRows": package["rawHtmlRows"]}
quality = [
    quality_row("SOURCE_POLICY_READY", "RAW_LINEAGE", "ERROR", "PASS" if recovery["rawHtmlRows"] == 29 else "FAIL", recovery["rawHtmlRows"], 29, "resume_state.json"),
    quality_row("OBSERVED_PACKAGE_REFERENCE_ONLY", "NO_RAW_COPY", "ERROR", "PASS" if package["rawCopied"] is False else "FAIL", package["rawCopied"], False, "observed_package/HANDOFF.json"),
]
persisted_files = [path for path in STAGE_ROOT.rglob("*") if path.is_file()]

## Termination artifacts and gate result

In [ ]:
from p4_crawl.stage import write_stage_artifacts

manifest = write_stage_artifacts(
    config=config, stage_id=STAGE_ID, schema_version=SCHEMA_VERSION,
    started_at=f"{AS_OF_DATE}T00:00:00+09:00", parameters=PARAMETERS,
    input_manifest_path=INPUT_MANIFEST, stage_root=STAGE_ROOT,
    metric_values=metrics, quality_rows=quality, persisted_files=persisted_files,
    warnings=[STAGE_WARNING], branch="agent/p4-crawl-release-v2",
)
if FAIL_ON_GATE and any(row["status"] == "FAIL" for row in quality):
    raise RuntimeError(f"{STAGE_ID} quality gate failed")
{"stageId": STAGE_ID, "status": manifest["status"], "metrics": metrics, "artifacts": manifest["terminationArtifacts"]}

Observed-development interpretation: Observed input is partial and cannot be promoted to empirical analysis. Analysis and production readiness remain disabled.